In [ ]:
import os
import time
import uuid
import json
import requests
from pdf2image import convert_from_path
from PIL import Image
from tqdm import tqdm

# ✅ Clova OCR 설정
secret_key = "WkliT2xxd2xSS1BUWkpjYWxSSm1rRGNUTHZSQkNLSGo="
api_url = "https://kvcj61qh9y.apigw.ntruss.com/custom/v1/44675/24aa45cfe5c7cbf30dd2f211eff37a5013dbb49a98d89d311258d50c1b072ef7/general"

# ✅ PDF → 이미지 변환
pdf_path = "../report/20241204_아마존 (AMZN US_매수).pdf"
pages = convert_from_path(pdf_path, dpi=300, poppler_path=r"C:\poppler-24.08.0\Library\bin")

ocr_results = []
temp_files = []  

for idx, page in enumerate(tqdm(pages, desc="OCR 진행 중")):
    temp_img_path = f"temp_page_{idx+1}.jpg"
    page.save(temp_img_path, "JPEG")
    temp_files.append(temp_img_path) 

    request_json = {
        'images': [{'format': 'jpg', 'name': f'page_{idx+1}'}],
        'requestId': str(uuid.uuid4()),
        'version': 'V2',
        'timestamp': int(round(time.time() * 1000))
    }

    payload = {'message': json.dumps(request_json).encode('UTF-8')}
    headers = {'X-OCR-SECRET': secret_key}

    try:
        with open(temp_img_path, 'rb') as f:
            files = [('file', f)]
            response = requests.post(api_url, headers=headers, data=payload, files=files, timeout=30)
    except requests.exceptions.Timeout:
        print(f" Page {idx+1} 연결 시간 초과")
        continue

    if response.status_code != 200:
        print(f" Page {idx+1} 실패 - 상태코드: {response.status_code}")
        continue

    result = response.json()
    page_text = ""
    for field in result.get("images", [])[0].get("fields", []):
        page_text += field.get("inferText", "") + " "

    ocr_results.append({
        "page": idx + 1,
        "text": page_text.strip()
    })


OCR 진행 중: 100%|██████████| 3/3 [00:08<00:00,  2.69s/it]


In [ ]:
# 결과 저장
full_text = "\n\n".join([f"[Page {r['page']}]\n{r['text']}" for r in ocr_results])
with open("ocr_result_clova_아마존_20241204.txt", "w", encoding="utf-8") as f:
    f.write(full_text)

print("OCR 결과 저장 완료.")

✅ OCR 결과 저장 완료.
[⚠️] 파일 삭제 실패 시도 1: temp_page_1.jpg → [WinError 32] 다른 프로세스가 파일을 사용 중이기 때문에 프로세스가 액세스 할 수 없습니다: 'temp_page_1.jpg'
[⚠️] 파일 삭제 실패 시도 2: temp_page_1.jpg → [WinError 32] 다른 프로세스가 파일을 사용 중이기 때문에 프로세스가 액세스 할 수 없습니다: 'temp_page_1.jpg'
[⚠️] 파일 삭제 실패 시도 3: temp_page_1.jpg → [WinError 32] 다른 프로세스가 파일을 사용 중이기 때문에 프로세스가 액세스 할 수 없습니다: 'temp_page_1.jpg'
[🚫] 파일 삭제 최종 실패: temp_page_1.jpg
[✅] 이미 삭제된 파일: temp_page_2.jpg
[✅] 이미 삭제된 파일: temp_page_3.jpg
